# RealMeta — Video to 3D Walkthrough (Live Demo)

This notebook runs RealMeta's video-to-3D pipeline on Google Colab's **free GPU**, so you can demo the real "upload a video → get a 3D model" process without needing to rent or manage a server.

**How to use this notebook:**
1. Click **Runtime → Change runtime type** and make sure **T4 GPU** is selected, then **Save**.
2. Run each cell below, top to bottom, by clicking the ▶️ button on the left of each cell (or press Shift+Enter).
3. When you reach the upload cell, you'll be prompted to upload your walkthrough video.
4. Wait for processing — this genuinely takes 15–40 minutes depending on video length. This is normal, not a bug.
5. The final cell gives you a download link and a live preview of your `.ply` result.

**Important limitations of the free tier (be upfront about this if demoing live):**
- Free Colab GPUs are shared and can occasionally be unavailable at busy times — if so, just try again in a few minutes.
- A free session can disconnect after a period of inactivity or after several hours — fine for a one-off demo, not for a real production backend (that's what the `/backend` folder + a rented GPU server is for).
- Processing time varies a lot with video length and lighting — a short, well-lit, slow walkthrough works best.

## Step 1 — Confirm we have a GPU

In [ ]:
!nvidia-smi

If this shows an error instead of a table with GPU info, go back to **Runtime → Change runtime type** and select a GPU, then re-run this cell.

## Step 2 — Install the tools the pipeline needs
(This takes a few minutes — COLMAP and the Gaussian Splatting code are being installed.)

In [ ]:
!apt-get -qq install -y colmap ffmpeg imagemagick > /dev/null
!pip -q install plyfile tqdm opencv-python-headless
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git /content/gaussian-splatting -q
%cd /content/gaussian-splatting
!pip -q install submodules/diff-gaussian-rasterization submodules/simple-knn
%cd /content
print('Setup complete.')

## Step 3 — Upload your walkthrough video
Click "Choose Files" below and select an .mp4 or .mov from your computer.

In [ ]:
from google.colab import files
import os

os.makedirs('/content/input', exist_ok=True)
uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
video_path = f'/content/input/{video_filename}'
os.rename(video_filename, video_path)
print(f'Uploaded: {video_path}')

## Step 4 — Extract frames from the video

In [ ]:
import os

project_dir = '/content/project'
images_dir = f'{project_dir}/images'
os.makedirs(images_dir, exist_ok=True)

fps = 2  # frames per second to pull from the video — raise this for a longer/slower walkthrough
!ffmpeg -y -i "$video_path" -qscale:v 1 -qmin 1 -vf fps=$fps "$images_dir/frame_%04d.jpg"

frame_count = len(os.listdir(images_dir))
print(f'Extracted {frame_count} frames.')
if frame_count < 20:
    print('WARNING: fewer than 20 frames — the next step (camera position estimation) works much better with more overlapping views. Consider a longer or slower walkthrough video.')

## Step 5 — Figure out where the camera was for each frame (COLMAP)
This step can take several minutes depending on how many frames you have.

In [ ]:
database_path = f'{project_dir}/database.db'
sparse_dir = f'{project_dir}/sparse'
os.makedirs(sparse_dir, exist_ok=True)

!colmap feature_extractor --database_path "$database_path" --image_path "$images_dir"
!colmap exhaustive_matcher --database_path "$database_path"
!colmap mapper --database_path "$database_path" --image_path "$images_dir" --output_path "$sparse_dir"

print('COLMAP reconstruction complete.')

## Step 6 — Train the 3D Gaussian Splat model
This is the longest step — typically 15–30+ minutes on a free Colab GPU. Feel free to leave the tab open and check back.

In [ ]:
training_dir = '/content/training'
iterations = 7000  # a good balance of speed vs quality for a demo; raise to 30000 for higher quality (much slower)

!python /content/gaussian-splatting/train.py -s "$project_dir" -m "$training_dir" --iterations $iterations

## Step 7 — Download your finished .ply file

In [ ]:
import glob
from google.colab import files as colab_files

candidates = sorted(glob.glob(f'{training_dir}/point_cloud/iteration_*/point_cloud.ply'))
if not candidates:
    print('No result found — check the training step above for errors.')
else:
    result_path = candidates[-1]
    print(f'Result ready: {result_path}')
    colab_files.download(result_path)

## Next step

Take the downloaded `.ply` file and load it in the RealMeta viewer webpage using the **"Load .ply directly instead"** link — this shows the same result you'd eventually get automatically once the `/backend` pipeline (see the main project folder) is deployed to a dedicated GPU server for real customers.